# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display all available record set @ids and their fields.
if hasattr(dataset, 'record_sets'):
    record_sets = dataset.record_sets
else:
    record_sets = dataset._record_sets  # fallback for some versions

print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# For each record set, print their available field/column @ids
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} - {rs.get('name', 'N/A')}")
    # Each record set may have a 'field' or 'column' element
    fields = rs.get('field', [])
    columns = rs.get('column', [])
    # 'field' and 'column' can be a dict (for single) or a list
    if isinstance(fields, dict):
        fields = [fields]
    if isinstance(columns, dict):
        columns = [columns]
    # Print fields
    for f in fields:
        # f can be either @id or dict
        if isinstance(f, str):
            print(f"  Field @id: {f}")
        elif isinstance(f, dict) and '@id' in f:
            print(f"  Field @id: {f['@id']}")
    # Print columns
    for c in columns:
        if isinstance(c, str):
            print(f"  Column @id: {c}")
        elif isinstance(c, dict) and '@id' in c:
            print(f"  Column @id: {c['@id']}")

# For illustration: Get the first record set's @id
selected_record_set_id = record_sets[0]['@id'] if record_sets else None
print(f"\nSelected record set for further use: {selected_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from available record sets.

# Get a list of all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records_generator = dataset.records(record_set=record_set_id)
    records = list(records_generator)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id}, shape: {df.shape}")
    else:
        print(f"No records found for record set @id: {record_set_id}")

# For demonstration, use the first loaded record set
main_record_set_id = None
for rsid in record_set_ids:
    if rsid in dataframes:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"\nMain DataFrame columns for {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set DataFrame loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA for an example numeric field in the selected record set.
import numpy as np

# Choose a numeric field (by @id or column name)
numeric_field_candidates = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype in [np.float64, np.int64, float, int]]
if not numeric_field_candidates:
    # Try to coerce one field to numeric
    for col in dataframes[main_record_set_id].columns:
        try:
            coerced = pd.to_numeric(dataframes[main_record_set_id][col], errors='coerce')
            if coerced.notnull().sum() > 0:
                dataframes[main_record_set_id][col] = coerced
                numeric_field_candidates.append(col)
                break
        except Exception:
            continue

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    threshold = dataframes[main_record_set_id][numeric_field].mean() if dataframes[main_record_set_id][numeric_field].mean() > 0 else 1
    print(f"Using numeric field: {numeric_field}, threshold: {threshold:.2f}")

    # Filter
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find categorical/group candidates
    possible_group_fields = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype == object]
    group_field = possible_group_fields[0] if possible_group_fields else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group/categorical field found for grouping.")
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple histogram and boxplot visualization using matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id and numeric_field_candidates:
    num_field = numeric_field_candidates[0]
    df = dataframes[main_record_set_id]
    df_valid = df[df[num_field].notnull()]
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(df_valid[num_field], bins=30, color='skyblue')
    plt.title(f"Distribution of {num_field}")
    plt.xlabel(num_field)
    plt.ylabel("Frequency")

    plt.subplot(1, 2, 2)
    plt.boxplot(df_valid[num_field], vert=False)
    plt.title(f"Boxplot of {num_field}")
    plt.xlabel(num_field)
    plt.tight_layout()
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        df_valid.boxplot(column=num_field, by=group_field, grid=False)
        plt.title(f"{num_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(num_field)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded the dataset metadata and main record sets using `mlcroissant` referencing all entities by their `@id`.
- Reviewed available fields and columns for data exploration.
- Performed basic EDA including filtering and normalization for selected numeric fields, and grouped data by categorical features (where present).
- Visualized distributions to identify trends or outliers in the selected measurements.

**Next steps:** Further scientific analysis, modeling, or integration of this dataset with biological or clinical research pipelines.